##### Copyright 2020 The Cirq Developers

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os, re, sys
from collections import defaultdict
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

drive.mount('/content/drive')

# Locate dataset folder in your Drive
base_dir = None
for root, dirs, files in os.walk('/content/drive/MyDrive/AMAZON'):
    if 'test_source1.tsv' in files:
        base_dir = os.path.dirname(os.path.dirname(root)) # points to student_resource root
        break

if not base_dir:
    # Default direct path from your Drive structure
    base_dir = '/content/drive/MyDrive/AMAZON/6ab10eb3b23ba_student_resource-1/student_resource'

test_dir = os.path.join(base_dir, 'dataset', 'test')
output_dir = os.path.join(base_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

print(f"Working with test directory: {test_dir}")
print(f"Output directory: {output_dir}")

# --- Normalization Rules ---
LEGAL_SUFFIXES = {
    r'\bpvt\.?\s*ltd\.?\b': 'private limited',
    r'\bltd\.?\b': 'limited',
    r'\binc\.?\b': 'incorporated',
    r'\bcorp\.?\b': 'corporation',
    r'\bllc\.?\b': 'llc',
    r'\bllp\.?\b': 'llp',
    r'\bco\.?\b': 'company',
    r'\bgmbh\b': 'gmbh',
    r'\bsarl\b': 'sarl',
    r'\bsas\b': 'sas',
    r'\bsa\b': 'sa',
}

ADDRESS_ABBREVIATIONS = {
    r'\brd\.?\b': 'road',
    r'\bst\.?\b': 'street',
    r'\bave\.?\b': 'avenue',
    r'\bblvd\.?\b': 'boulevard',
    r'\bln\.?\b': 'lane',
    r'\bdr\.?\b': 'drive',
    r'\bct\.?\b': 'court',
    r'\bste\.?\b': 'suite',
    r'\bapt\.?\b': 'apartment',
    r'\bfl\.?\b': 'floor',
    r'\bdist\.?\b': 'district',
}

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = s.lower().strip()
    s = re.sub(r'[\t\r\n]+', ' ', s)
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def normalize_name(name):
    text = clean_text(name)
    for pattern, repl in LEGAL_SUFFIXES.items():
        text = re.sub(pattern, repl, text)
    return re.sub(r'\s+', ' ', text).strip()

def normalize_address(addr):
    text = clean_text(addr)
    for pattern, repl in ADDRESS_ABBREVIATIONS.items():
        text = re.sub(pattern, repl, text)
    return re.sub(r'\s+', ' ', text).strip()

# --- Execution ---
print("Reading test sources...")
s1 = pd.read_csv(os.path.join(test_dir, "test_source1.tsv"), sep="\t", dtype=str).fillna("")
s2 = pd.read_csv(os.path.join(test_dir, "test_source2.tsv"), sep="\t", dtype=str).fillna("")
s3 = pd.read_csv(os.path.join(test_dir, "test_source3.tsv"), sep="\t", dtype=str).fillna("")

targets = pd.concat([s2, s3], ignore_index=True)

print("Normalizing records...")
s1['comb'] = s1['business_name'].apply(normalize_name) + " " + s1['business_address'].apply(normalize_address)
s1['country_clean'] = s1['country'].str.strip().str.upper()

targets['comb'] = targets['business_name'].apply(normalize_name) + " " + targets['business_address'].apply(normalize_address)
targets['country_clean'] = targets['country'].str.strip().str.upper()

candidate_dict = defaultdict(list)
match_dict = defaultdict(list)

# Block by country to scale across millions of pairs
for c in s1['country_clean'].unique():
    print(f"Processing country: {c}")
    sub_s1 = s1[s1['country_clean'] == c].reset_index(drop=True)
    sub_tgt = targets[targets['country_clean'] == c].reset_index(drop=True)

    if len(sub_tgt) == 0:
        for eid in sub_s1['entity_id']:
            candidate_dict[eid] = []
            match_dict[eid] = []
        continue

    # Character n-grams (3-4) for typo tolerance and legal variant matching
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), min_df=2)
    vec.fit(pd.concat([sub_s1['comb'], sub_tgt['comb']]))

    s1_vecs = vec.transform(sub_s1['comb'])
    tgt_vecs = vec.transform(sub_tgt['comb'])

    batch_size = 500
    for b_idx in range(0, len(sub_s1), batch_size):
        chunk = s1_vecs[b_idx : b_idx + batch_size]
        sims = cosine_similarity(chunk, tgt_vecs)

        for local_i, row in enumerate(sims):
            global_i = b_idx + local_i
            s1_id = sub_s1.at[global_i, 'entity_id']
            top_k = np.argsort(row)[::-1][:25]

            cands = []
            matches = []
            for t_idx in top_k:
                score = row[t_idx]
                if score < 0.20:
                    break
                tid = sub_tgt.at[t_idx, 'entity_id']
                cands.append(tid)
                if score >= 0.72:  # High threshold tuned for precision-heavy F_0.5
                    matches.append(tid)

            candidate_dict[s1_id] = cands
            match_dict[s1_id] = matches

# Write Output TSVs
cand_rows = [(eid, ",".join(candidate_dict[eid])) for eid in s1['entity_id']]
match_rows = [(eid, ",".join(match_dict[eid])) for eid in s1['entity_id']]

cand_path = os.path.join(output_dir, "candidate_pairs.tsv")
match_path = os.path.join(output_dir, "matching_results.tsv")

pd.DataFrame(cand_rows, columns=['source1_entity_id', 'candidate_entity_ids']).to_csv(cand_path, sep="\t", index=False)
pd.DataFrame(match_rows, columns=['source1_entity_id', 'matched_entity_ids']).to_csv(match_path, sep="\t", index=False)

print(f"Generated: {match_path}")
print(f"Generated: {cand_path}")

# Run official validator
validate_script = os.path.join(base_dir, 'utils', 'validate_submission.py')
if os.path.exists(validate_script):
    !python3 "{validate_script}" --matching "{match_path}" --candidate "{cand_path}" --test-dir "{test_dir}"